# Exercise 7.2: Sampling calorimeter

From *Programming in High Energy Particle Physics*, Chapter 7

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch07/ex07_2_solution.ipynb)

Exercise 7.2 Sampling calorimeter Simulate a sampling calorimeter with alternating layers of lead (absorber) and plastic scintillator (active medium). Vary the absorber-to-scintillator ratio and study how the energy resolution (\(\sigma_E/E\)) and sampling fraction change.

<details><summary>Hint</summary>

Record the energy in the scintillator and in the lead separately. The sampling fraction is \(E_\text{scint}/(E_\text{scint}+E_\text{Pb})\). Fit the response at several beam energies with Eq. 7.3. A useful empirical rule for electromagnetic sampling calorimeters is \(a_s \approx 2.7\%\sqrt{d/f_\text{samp}}\), with \(d\) the thickness of the active layers in mm and \(f_\text{samp}\) the sampling fraction for minimum-ionizing particles.

</details>

### Setup

The Colab analysis is an **empirical stand-in**, not Geant4 shower transport. `geant4_pybind` is an optional heavy download, including large datasets; a full C++ Geant4 build is required for the simulation below.

In [ ]:
import sys
IN_COLAB="google.colab" in sys.modules
if IN_COLAB:
    %pip install -q numpy scipy matplotlib
INSTALL_GEANT4_BINDINGS=False
if IN_COLAB and INSTALL_GEANT4_BINDINGS:
    %pip install -q geant4_pybind
import numpy as np
import matplotlib.pyplot as plt
rng=np.random.default_rng(42)

### Complete local Geant4 shower simulation

The code records active and absorber energy separately. It varies Pb layer thickness while holding the total Pb near 120 mm (about 21 X₀), so changes in shower containment are reduced; the number of layers and calorimeter length change.

In [ ]:
%%writefile ex07_2.cc
#include "G4RunManager.hh"
#include "G4VUserDetectorConstruction.hh"
#include "G4VUserPrimaryGeneratorAction.hh"
#include "G4VUserActionInitialization.hh"
#include "G4UserEventAction.hh"
#include "G4UserSteppingAction.hh"
#include "G4NistManager.hh"
#include "G4Box.hh"
#include "G4LogicalVolume.hh"
#include "G4PVPlacement.hh"
#include "G4ParticleGun.hh"
#include "G4ParticleTable.hh"
#include "G4SystemOfUnits.hh"
#include "G4Event.hh"
#include "G4Step.hh"
#include "FTFP_BERT.hh"
#include "Randomize.hh"
#include <fstream>
#include <string>
#include <cstdlib>
class Calorimeter : public G4VUserDetectorConstruction {
  double leadMM;
public:
  G4LogicalVolume *lead=nullptr,*scint=nullptr;
  double startZ=0;
  explicit Calorimeter(double d):leadMM(d){}
  G4VPhysicalVolume* Construct() override {
    auto* nist=G4NistManager::Instance();
    auto* air=nist->FindOrBuildMaterial("G4_AIR");
    auto* pb=nist->FindOrBuildMaterial("G4_Pb");
    auto* plastic=nist->FindOrBuildMaterial("G4_POLYSTYRENE");
    auto* world=new G4LogicalVolume(new G4Box("world",50*cm,50*cm,1*m),air,"world");
    auto* pv=new G4PVPlacement(nullptr,{},world,"world",nullptr,false,0);
    const int n=static_cast<int>(120.0/leadMM); // same ~120 mm Pb = ~21 X0
    const double scintMM=2.0;
    const double pitch=(leadMM+scintMM)*mm;
    startZ=-0.5*n*pitch;
    lead=new G4LogicalVolume(new G4Box("Pb",5*cm,5*cm,0.5*leadMM*mm),pb,"Pb");
    scint=new G4LogicalVolume(new G4Box("Scint",5*cm,5*cm,0.5*scintMM*mm),plastic,"Scint");
    for(int i=0;i<n;++i) {
      new G4PVPlacement(nullptr,G4ThreeVector(0,0,startZ+i*pitch+0.5*leadMM*mm),lead,"Pb",world,false,i);
      new G4PVPlacement(nullptr,G4ThreeVector(0,0,startZ+i*pitch+(leadMM+0.5*scintMM)*mm),scint,"Scint",world,false,i);
    }
    return pv;
  }
};
class ElectronBeam : public G4VUserPrimaryGeneratorAction {
  G4ParticleGun gun{1};
public:
  ElectronBeam(double energyGeV,double startZ) {
    gun.SetParticleDefinition(G4ParticleTable::GetParticleTable()->FindParticle("e-"));
    gun.SetParticleEnergy(energyGeV*GeV);
    gun.SetParticlePosition(G4ThreeVector(0,0,startZ-5*mm));
    gun.SetParticleMomentumDirection(G4ThreeVector(0,0,1));
  }
  void GeneratePrimaries(G4Event* e) override {gun.GeneratePrimaryVertex(e);}
};
class CalEvent : public G4UserEventAction {
  std::ofstream out;
  double eScint=0,ePb=0;
public:
  explicit CalEvent(const std::string& path):out(path) {out << "scint_MeV,pb_MeV\n";}
  void BeginOfEventAction(const G4Event*) override {eScint=ePb=0;}
  void AddScint(double e){eScint+=e;}
  void AddPb(double e){ePb+=e;}
  void EndOfEventAction(const G4Event*) override {out << eScint/MeV << ',' << ePb/MeV << '\n';}
};
class CalStep : public G4UserSteppingAction {
  const Calorimeter* det;
  CalEvent* evt;
public:
  CalStep(const Calorimeter* d,CalEvent* e):det(d),evt(e){}
  void UserSteppingAction(const G4Step* step) override {
    auto* lv=step->GetPreStepPoint()->GetTouchableHandle()->GetVolume()->GetLogicalVolume();
    if(lv==det->scint) evt->AddScint(step->GetTotalEnergyDeposit());
    if(lv==det->lead) evt->AddPb(step->GetTotalEnergyDeposit());
  }
};
class Actions : public G4VUserActionInitialization {
  const Calorimeter* det;
  double energy;
  std::string path;
public:
  Actions(const Calorimeter* d,double e,std::string p):det(d),energy(e),path(p){}
  void Build() const override {
    SetUserAction(new ElectronBeam(energy,det->startZ));
    auto* evt=new CalEvent(path);SetUserAction(evt);
    SetUserAction(new CalStep(det,evt));
  }
};
int main(int argc,char** argv) {
  if(argc<5) return 2; // Pb thickness mm, energy GeV, events, CSV
  const double leadMM=std::atof(argv[1]);
  if(leadMM<=0 || leadMM>5) return 3;
  G4Random::setTheSeed(42);
  auto* rm=new G4RunManager;
  auto* det=new Calorimeter(leadMM);
  rm->SetUserInitialization(det);
  rm->SetUserInitialization(new FTFP_BERT);
  rm->SetUserInitialization(new Actions(det,std::atof(argv[2]),argv[4]));
  rm->Initialize();rm->BeamOn(std::atoi(argv[3]));delete rm;
}


In [ ]:
%%writefile CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(ex07_2 LANGUAGES CXX)
find_package(Geant4 REQUIRED)
include(${Geant4_USE_FILE})
add_executable(ex07_2 ex07_2.cc)
target_link_libraries(ex07_2 ${Geant4_LIBRARIES})


Local commands after extracting both files:

```bash
cmake -S . -B build
cmake --build build -j2
./build/ex07_2 1 1 500 pb1_E1.csv
./build/ex07_2 2 1 500 pb2_E1.csv
./build/ex07_2 4 1 500 pb4_E1.csv
```

Repeat at beam energies 2, 5 and 10 GeV; compare `scint_MeV/(scint_MeV+pb_MeV)` and the standard deviation of calibrated scintillator energy divided by its mean. Geant4 datasets must be configured locally. The empirical Colab calculation below runs without Geant4.
If CMake cannot locate an optional dependency in an existing Geant4 installation, a direct build is also possible with `c++ ex07_N.cc -o ex07_N $(geant4-config --cflags --libs)` (replace `N` with this exercise number and configure the library path for that installation).

### Colab stand-in: empirical sampling response

The MIP sampling fraction uses representative stopping-power factors for polystyrene and lead. The stochastic term uses the empirical formula in the exercise. Gaussian shower fluctuations are imposed; there is no particle transport or leakage model here.

In [ ]:
from scipy.optimize import curve_fit
lead_mm=np.array([1.,2.,4.])
scint_mm=2.0
# Approximate MIP mass stopping powers [MeV cm²/g] and densities [g/cm³].
scint_loss=2.0*1.05*scint_mm/10
lead_loss=1.12*11.34*lead_mm/10
f_samp=scint_loss/(scint_loss+lead_loss)
a=0.027*np.sqrt(scint_mm/f_samp)
energies=np.array([1.,2.,5.,10.]) # GeV
noise=0.02 # GeV
constant=0.01
def resolution(E,a,b,c):return np.sqrt(a*a/E+b*b/E**2+c*c)
measured=np.empty((len(lead_mm),len(energies)))
for i,f in enumerate(f_samp):
    for j,E in enumerate(energies):
        response=rng.normal(f*E,f*E*resolution(E,a[i],noise,constant),size=4000)
        measured[i,j]=response.std(ddof=1)/response.mean()
print("Pb thickness [mm]:",lead_mm)
print("MIP sampling fraction:",np.round(f_samp,3))
print("stochastic a [fraction sqrt(GeV)]:",np.round(a,3))
assert np.all(np.diff(f_samp)<0)

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(lead_mm,f_samp,"o-")
ax[0].set(xlabel="Pb per layer [mm]",ylabel="MIP sampling fraction")
for i,d in enumerate(lead_mm):
    pars,_=curve_fit(resolution,energies,measured[i],p0=[a[i],noise,constant],bounds=(0,[1,1,1]))
    ax[1].plot(energies,measured[i],"o",label=f"{d:g} mm Pb (toy)")
    grid=np.linspace(1,10,150)
    ax[1].plot(grid,resolution(grid,*pars),"--")
    print(f"{d:g} mm Pb: fitted a≈{pars[0]:.3f} sqrt(GeV)")
ax[1].set(xlabel="Beam energy [GeV]",ylabel=r"$\sigma_E/E$")
ax[1].legend();plt.tight_layout();plt.show()

Thinner Pb layers raise the MIP sampling fraction and lower the stochastic term, consistent with the book. At fixed absorber radiation length this requires more layers and a longer detector. The stand-in predicts that trend by construction; only the local Geant4 run can test shower fluctuations, leakage, and whether the empirical formula describes a particular geometry.

### Analyze local Geant4 shower CSVs when available

In [ ]:
from pathlib import Path
records=[]
for d in lead_mm:
    for E in energies:
        p=Path(f"pb{d:g}_E{E:g}.csv")
        if not p.exists():continue
        x=np.genfromtxt(p,delimiter=",",names=True)
        scint=np.atleast_1d(x["scint_MeV"]);pb=np.atleast_1d(x["pb_MeV"])
        frac=np.mean(scint/(scint+pb))
        rel=scint.std(ddof=1)/scint.mean()
        records.append((d,E,frac,rel,len(scint)))
        print(f"Pb {d:g} mm, E {E:g} GeV: f={frac:.3f}, σ/mean={rel:.3f}, N={len(scint)}")
if records:
    fig,ax=plt.subplots()
    for d in lead_mm:
        rows=sorted((r for r in records if r[0]==d),key=lambda r:r[1])
        if rows:
            ax.plot([r[1] for r in rows],[r[3] for r in rows],"o-",label=f"{d:g} mm Pb")
            if len(rows)>=4:
                pars,_=curve_fit(resolution,[r[1] for r in rows],[r[3] for r in rows],p0=[0.1,0.02,0.01],bounds=(0,[1,1,1]))
                grid=np.linspace(min(r[1] for r in rows),max(r[1] for r in rows),100)
                ax.plot(grid,resolution(grid,*pars),"--")
                print(f"Pb {d:g} mm fit: a={pars[0]:.3f} sqrt(GeV), b={pars[1]:.3f} GeV, c={pars[2]:.3f}")
    ax.set(xlabel="Beam energy [GeV]",ylabel=r"$\sigma_E/E$ (scintillator response)");ax.legend();plt.show()
else:
    print("Run the local Geant4 program at several Pb thicknesses and energies, then place the CSVs beside this notebook.")